In [43]:
import os
import glob
import pandas as pd
import geopandas as gpd

file_inputs = [
    {"path": "data/alabama2026.geojson", "postal": "AL"},
    {"path": "data/tennessee2026.geojson", "postal": "TN"} 
]

# Standard target CRS (EPSG:4326 is standard for GeoJSON)
TARGET_CRS = "EPSG:4326"
all_gdfs = []

for state in file_inputs:
    gdf = gpd.read_file(state["path"])

    if gdf.crs != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)
        
    # Create matching key for the geojson/csv files with state postal and district number
    gdf["District"] = state["postal"] + gdf["id"].astype(str).str.zfill(2)
    
    gdf = gdf[["id", "District", "DemPct", "RepPct", "Margin","geometry"]]
    
    all_gdfs.append(gdf)

states_gdf = pd.concat(all_gdfs, ignore_index=True)
states_gdf.head(7)

,id,District,DemPct,RepPct,Margin,geometry
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ..."
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ..."
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ..."
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ..."
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ..."
5,6,AL06,0.320528,0.662526,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ..."
6,7,AL07,0.582883,0.405147,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ..."


In [44]:
# 2024 presidential results by Congressional district
pres24 = pd.read_csv("data/2024_pres.csv")

# Merged desired columns from the csv file into the geojson file
columns_to_keep = ['District', 'Harris', 'Trump 24', 'Total',
       'Harris %', 'Trump %', 'Margin']

pres24 = pres24[columns_to_keep]

# Inner join the datasets on matching District values
merged_gdf = states_gdf.merge(pres24, on='District', how='inner')
merged_gdf.head(7)

,id,District,DemPct,RepPct,Margin_x,geometry,Harris,Trump 24,Total,Harris %,Trump %,Margin_y
0,1,AL01,0.314327,0.673261,-0.185673,"MULTIPOLYGON (((-86.76351 31.18113, -86.76637 ...",73003,257060,332700,21.94%,77.26%,-55.32%
1,2,AL02,0.422827,0.565831,-0.077173,"MULTIPOLYGON (((-85.10754 31.18984, -85.10744 ...",155603,131721,290033,53.65%,45.42%,8.23%
2,3,AL03,0.261882,0.727057,-0.238118,"MULTIPOLYGON (((-85.13504 32.74653, -85.13616 ...",82654,229676,314869,26.25%,72.94%,-46.69%
3,4,AL04,0.163376,0.826469,-0.336624,"MULTIPOLYGON (((-87.30474 34.29946, -87.31559 ...",53098,267953,323449,16.42%,82.84%,-66.43%
4,5,AL05,0.346103,0.635919,-0.153897,"MULTIPOLYGON (((-87.42457 34.75436, -87.42560 ...",122344,224704,351650,34.79%,63.90%,-29.11%
5,6,AL06,0.320528,0.662526,-0.179472,"MULTIPOLYGON (((-86.33906 32.57720, -86.33884 ...",105388,239984,349125,30.19%,68.74%,-38.55%
6,7,AL07,0.582883,0.405147,0.082883,"MULTIPOLYGON (((-87.58573 33.08732, -87.58590 ...",180321,111517,294526,61.22%,37.86%,23.36%
